###  Set Up OAuth Authentication

In [0]:
# Retrieve secrets from Databricks Secret Scope
# Secrets are never displayed in output — shown as [REDACTED]
client_id     = dbutils.secrets.get(scope="africa-pulse-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="africa-pulse-scope", key="client-secret")
tenant_id     = dbutils.secrets.get(scope="africa-pulse-scope", key="tenant-id")
storage_acct  = dbutils.secrets.get(scope="africa-pulse-scope", key="storage-account")

# Configure OAuth
spark.conf.set(f"fs.azure.account.auth.type.{storage_acct}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_acct}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_acct}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_acct}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_acct}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("Authentication configured successfully")
print(f"Client ID: {client_id[:8]}...") # Shows only first 8 chars to confirm it loaded

### Define Paths

In [0]:
# Define storage paths
STORAGE_ACCOUNT = "africapulse"
CONTAINER = "africa-economic-pulse"
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Bronze paths (source)
BRONZE_GDP_PATH = f"{BASE_PATH}/bronze/world_bank/NY.GDP.MKTP.CD/"
BRONZE_INFLATION_PATH = f"{BASE_PATH}/bronze/world_bank/FP.CPI.TOTL.ZG/"
BRONZE_UNEMPLOYMENT_PATH = f"{BASE_PATH}/bronze/world_bank/SL.UEM.TOTL.ZS/"

# Silver paths (destination)
SILVER_GDP_PATH = f"{BASE_PATH}/silver/gdp_cleaned/"
SILVER_INFLATION_PATH = f"{BASE_PATH}/silver/inflation_cleaned/"
SILVER_UNEMPLOYMENT_PATH = f"{BASE_PATH}/silver/unemployment_cleaned/"

print("Paths configured:")
print(f"Bronze GDP: {BRONZE_GDP_PATH}")
print(f"Silver GDP: {SILVER_GDP_PATH}")

### Verify Bronze Data Landed

In [0]:
import requests
import json
from pyspark.sql import Row

# All 54 African country ISO3 codes
african_countries = "DZA;AGO;BEN;BWA;BFA;BDI;CPV;CMR;CAF;TCD;COM;COD;COG;CIV;DJI;EGY;GNQ;ERI;SWZ;ETH;GAB;GMB;GHA;GIN;GNB;KEN;LSO;LBR;LBY;MDG;MWI;MLI;MRT;MUS;MAR;MOZ;NAM;NER;NGA;RWA;STP;SEN;SLE;SOM;ZAF;SSD;SDN;TZA;TGO;TUN;UGA;ZMB;ZWE"

indicators = {
    "gdp": "NY.GDP.MKTP.CD",
    "inflation": "FP.CPI.TOTL.ZG",
    "unemployment": "SL.UEM.TOTL.ZS"
}

def fetch_worldbank_data(countries, indicator_code):
    """
    Fetch World Bank indicator data for specified countries.
    Returns list of flat record dictionaries.
    """
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{indicator_code}"
    params = {
        "format": "json",
        "per_page": 1000,
        "mrv": 25
    }
    
    all_records = []
    page = 1
    
    while True:
        params["page"] = page
        response = requests.get(url, params=params)
        
        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break
            
        data = response.json()
        metadata = data[0]
        records = data[1]
        
        if not records:
            break
            
        for record in records:
            flat = {
                "indicator_id":    str(record.get("indicator", {}).get("id", "")),
                "indicator_name":  str(record.get("indicator", {}).get("value", "")),
                "country_id":      str(record.get("country", {}).get("id", "")),
                "country_name":    str(record.get("country", {}).get("value", "")),
                "country_iso3":    str(record.get("countryiso3code", "") or ""),
                "year":            str(record.get("date", "") or ""),
                "indicator_value": float(record["value"]) if record.get("value") is not None else None,
                "unit":            str(record.get("unit", "") or ""),
                "obs_status":      str(record.get("obs_status", "") or "")
            }
            all_records.append(flat)
        
        print(f"  Page {page}/{metadata['pages']} fetched — {len(records)} records")
        
        if page >= metadata["pages"]:
            break
        page += 1
    
    return all_records

# Fetch all three indicators
all_data = {}
for name, code in indicators.items():
    print(f"\nFetching {name} ({code})...")
    records = fetch_worldbank_data(african_countries, code)
    df = spark.createDataFrame(records)
    all_data[name] = df
    print(f"✅ {name}: {df.count()} records, {df.select('country_name').distinct().count()} countries")

# Verify Nigeria is present
print("\nNigeria GDP check:")
display(all_data["gdp"].filter(col("country_iso3") == "NGA").orderBy("year", ascending=False).limit(5))

### Parse and Flatten

In [0]:
from pyspark.sql.functions import col

# Verify Nigeria is present
print("\nNigeria GDP check:")
display(all_data["gdp"].filter(col("country_iso3") == "NGA").orderBy("year", ascending=False).limit(5))

### Full Silver transformation and write to Delta

In [0]:
from pyspark.sql.functions import col, lit, current_date, trim
from pyspark.sql.types import IntegerType

def clean_and_enrich(df):
    """Apply data quality rules and add metadata columns"""
    return df\
        .filter(col("indicator_value").isNotNull())\
        .filter(col("year").isNotNull())\
        .filter(col("country_iso3") != "")\
        .withColumn("year", col("year").cast(IntegerType()))\
        .withColumn("country_name", trim(col("country_name")))\
        .withColumn("ingestion_date", current_date())\
        .withColumn("pipeline_name", lit("worldbank_silver_transform"))\
        .withColumn("source_system", lit("World Bank API v2"))

# Write all three indicators to Silver as Delta tables
for name, df in all_data.items():
    silver_path = f"{BASE_PATH}/silver/{name}_cleaned/"
    
    cleaned = clean_and_enrich(df)
    
    cleaned.write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .partitionBy("year")\
        .save(silver_path)
    
    count = spark.read.format("delta").load(silver_path).count()
    print(f"✅ {name}: {count} records written to Silver at {silver_path}")

print("\nAll Silver tables written successfully!")